## MobileNetV2 Training Model for Plant Disease Detection. Project Codename (RHOMBUS)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize
import seaborn as sns
import pandas as pd
from datetime import datetime
import random
import cv2

In [ ]:
# ===============================
# 🔧 Configuration
# ===============================
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30
L2_REG = 0.0005
DROPOUT_RATE = 0.4
LEARNING_RATE = 1e-5
TRAIN_DATASET_PATH = 'data/PlantVillage/train'  
TEST_DATASET_PATH = 'data/PlantVillage/test'  
VALIDATION_DATASET_PATH = 'data/PlantVillage/val' 

In [ ]:
# ===============================
# 🔄 Data Augmentation
# ===============================
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.15,
    brightness_range=(0.8, 1.2),
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    TRAIN_DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    VALIDATION_DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    TEST_DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # important to preserve label order
)

In [ ]:
# ===============================
# 🧠 Load MobileNetV2 base
# ===============================
base_model = MobileNetV2(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # Freeze for transfer learning

# Unfreeze last N layers
for layer in base_model.layers[-30:]:
    layer.trainable = True

# ===============================
# 🏗️ Add Custom Classification Head
# ===============================
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(1024, activation='relu', kernel_regularizer=l2(L2_REG))(x)
x = Dropout(DROPOUT_RATE)(x)
NUM_CLASSES = len(list(train_gen.class_indices.keys()))
predictions = Dense(NUM_CLASSES, activation='softmax', kernel_regularizer=l2(L2_REG))(x)

model = Model(inputs=base_model.input, outputs=predictions)

# ===============================
# ⚙️ Compile
# ===============================
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy',
              metrics=['accuracy',tf.keras.metrics.AUC(name='auc')])

# ===============================
# 📌 Callbacks
# ===============================
checkpoint_cb = ModelCheckpoint("models/h5/best_rhombus_model.h5", monitor='val_accuracy', save_best_only=True, verbose=1)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, min_lr=1e-6)
earlystop_cb = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

In [ ]:
def shorten_labels(class_names, max_len=12):
    """
    Shortens long class names with ellipses for cleaner visualization.

    Args:
        class_names (list of str): Original class names.
        max_len (int): Maximum allowed length before truncation.

    Returns:
        list of str: Shortened class names.
    """
    shortened = []
    for name in class_names:
        if len(name) > max_len:
            shortened.append(name[:max_len - 3] + '...')
        else:
            shortened.append(name)
    return shortened

def shorten_label(label, max_len=30):
    """
    Shortens a single class label if it exceeds the max length.

    Args:
        label (str): The class name.
        max_len (int): Max allowed length before truncation.

    Returns:
        str: Shortened label with ellipsis if needed.
    """
    return label[:max_len - 3] + '...' if len(label) > max_len else label


In [ ]:

def show_random_images(data_dir, num_images=16, grid_size=(4, 4), save_path="eda_outputs"):
    """
    Display and save random images from random classes in a 4x4 grid.
    
    Args:
        data_dir (str): Path to training data directory (organized by class folders).
        num_images (int): Total number of images to display.
        grid_size (tuple): Grid dimensions (rows, cols).
        save_path (str): Directory to save the image grid as PNG.
    """
    os.makedirs(save_path, exist_ok=True)
    class_folders = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    images = []

    while len(images) < num_images:
        cls = random.choice(class_folders)
        class_dir = os.path.join(data_dir, cls)
        img_files = os.listdir(class_dir)
        if img_files:
            img_path = os.path.join(class_dir, random.choice(img_files))
            images.append((img_path, cls))

    # Plotting
    fig = plt.figure(figsize=(12, 12))
    for i, (img_path, label) in enumerate(images):
        plt.subplot(*grid_size, i + 1)
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.title(shorten_label(label), fontsize=9)
        plt.axis('off')
    
    plt.suptitle("Image Grid from Training Set", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    # Save the figure
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"random_grid_{timestamp}.png"
    fig_path = os.path.join(save_path, filename)
    plt.savefig(fig_path, dpi=300)
    print(f"✅ Grid saved as: {fig_path}")
    plt.show()

show_random_images(TRAIN_DATASET_PATH)


In [ ]:
# ===============================
# 🚀 Train
# ===============================
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=[checkpoint_cb, earlystop_cb, lr_reduce]
)

In [ ]:
print(f"\n✅ Best Validation Accuracy: {max(history.history['val_accuracy']) * 100:.2f}%")
print(f"\n✅ Best Validation Loss: {min(history.history['val_loss']) * 100:.2f}%")

In [ ]:
class_names = list(test_generator.class_indices.keys())
class_names = shorten_labels(class_names, max_len=20)
base_dir= "evaluation_results"
save_dir= os.path.join(base_dir, 'rhombus')
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# ===============================
# 📈 Plot Training Curves
# ===============================

# Plot training vs validation accuracy/loss
def plot_training_summary(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(11, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.savefig(os.path.join(save_dir, "rhombus_training_validation_plot.png"), dpi=300)
    plt.tight_layout()
    plt.show()

plot_training_summary(history)

In [ ]:
# ===============================
# 📈 Plot AUC Training Curves
# ===============================

# Plot training vs validation auc
def plot_auc(history):
    """
    Plots AUC for training and validation over epochs.
    """
    if 'auc' not in history.history:
        print("AUC metric not found in history. Please ensure 'tf.keras.metrics.AUC' was used during model.compile().")
        return

    train_auc = history.history['auc']
    val_auc = history.history['val_auc']
    epochs = range(1, len(train_auc) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_auc, label='Training AUC', marker='o')
    plt.plot(epochs, val_auc, label='Validation AUC', marker='x')
    plt.title('Training and Validation AUC')
    plt.xlabel('Epochs')
    plt.ylabel('AUC Score')
    plt.ylim(0.0, 1.05)
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, "rhombus_auc_training_validation_plot.png"), dpi=300)
    plt.legend()
    plt.tight_layout()
    plt.show()
plot_auc(history)

In [ ]:
    os.makedirs(save_dir, exist_ok=True)
    print("\n📊 Evaluating on test set...")
    saved_model = load_model("models/h5/best_rhombus_model.h5")
    test_loss, test_accuracy, test_auc = saved_model.evaluate(test_generator, verbose=1)
    print(f"\n✅ Test Accuracy: {test_accuracy * 100:.2f}%")
    print(f"🧪 Test Loss: {test_loss * 100:.2f}%")
    print(f"🧪 Test AUC: {test_auc * 100:.2f}%")

    print("\n🔍 Generating predictions...")
    y_true = test_generator.classes
    y_pred_probs = saved_model.predict(test_generator, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # === Classification Report ===
    print("\n📄 Classification Report:")
    report = classification_report(y_true, y_pred, target_names=class_names)
    print(report)

In [ ]:
    # === Confusion Matrix ===
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix - Test Set", fontsize=16)
    plt.xlabel("Predicted", fontsize=14)
    plt.ylabel("True", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.savefig(os.path.join(save_dir, "rhombus_confusion_matrix.png"))
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
    # === AUC-ROC ===
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    overall_auc = roc_auc_score(y_true_bin, y_pred_probs, average='macro', multi_class='ovr')
    print(f"\n🔵 Macro AUC-ROC Score: {overall_auc:.4f}")

    per_class_auc = []
    for i in range(len(class_names)):
        auc = roc_auc_score(y_true_bin[:, i], y_pred_probs[:, i])
        per_class_auc.append(auc)

    # === Per-Class Accuracy ===
    class_correct = np.zeros(len(class_names))
    class_total = np.zeros(len(class_names))
    for i in range(len(y_true)):
        label = y_true[i]
        pred = y_pred[i]
        class_total[label] += 1
        if label == pred:
            class_correct[label] += 1
    class_accuracy = class_correct / class_total

    # === Save CSV Report ===
    df_results = pd.DataFrame({
        'Class': class_names,
        'Accuracy': class_accuracy,
        'AUC_ROC': per_class_auc
    })
    df_results.to_csv(os.path.join(save_dir, "rhombus_per_class_metrics.csv"), index=False)

    # === Per-Class Accuracy Plot ===
    plt.figure(figsize=(14, 8))
    plt.barh(class_names, class_accuracy, color='mediumseagreen')
    plt.xlabel("Accuracy")
    plt.title("Accuracy Per-Class on Test Set")
    plt.grid(axis='x', linestyle='--')
    plt.savefig(os.path.join(save_dir, "rhombus_per_class_accuracy.png"))
    plt.tight_layout()
    plt.show()
    plt.close()

    # === Per-Class AUC-ROC Plot ===
    plt.figure(figsize=(14, 8))
    plt.barh(class_names, per_class_auc, color='steelblue')
    plt.xlabel("AUC-ROC")
    plt.title("AUC-ROC Per-Class on Test Set")
    plt.grid(axis='x', linestyle='--')
    plt.savefig(os.path.join(save_dir, "per_class_auc.png"))
    plt.tight_layout()
    plt.show()
    plt.close()

    # === Save full summary text ===
    with open(os.path.join(save_dir, "rhombus_summary.txt"), "w") as f:
        f.write(f"Test Accuracy: {test_accuracy:.4f}\n")
        f.write(f"Test Loss: {test_loss:.4f}\n")
        f.write(f"AUC-ROC (macro avg): {overall_auc:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(report)

    print(f"\n📁 All evaluation results saved to: {save_dir}")

In [ ]:
# Grad-CAM function
def generate_gradcam(model, img_array, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Overlay heatmap on image
def overlay_heatmap(heatmap, image_orig, alpha=0.4):
    heatmap = cv2.resize(heatmap, (image_orig.shape[1], image_orig.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(heatmap_color, alpha, image_orig, 1 - alpha, 0)
    return overlay

# Grad-CAM on multiple samples
def gradcam_on_multiple_images(model, test_dir, conv_layer_name, num_images=6, target_size=(224, 224)):
    class_folders = [os.path.join(test_dir, c) for c in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, c))]
    images = []

    while len(images) < num_images:
        cls_dir = random.choice(class_folders)
        img_file = random.choice(os.listdir(cls_dir))
        img_path = os.path.join(cls_dir, img_file)
        images.append((img_path, os.path.basename(cls_dir)))

    plt.figure(figsize=(12, num_images * 2))

    for i, (img_path, label) in enumerate(images):
        img = image.load_img(img_path, target_size=target_size)
        img_array = image.img_to_array(img) / 255.0
        img_batch = np.expand_dims(img_array, axis=0)
        original_label = shorten_label(label, max_len=18)
        grad_cam_label = shorten_label(label, max_len=15)

        # Generate Grad-CAM heatmap
        heatmap = generate_gradcam(model, img_batch, last_conv_layer_name=conv_layer_name)
        overlay = overlay_heatmap(heatmap, np.uint8(img_array * 255))

        # Plot original + Grad-CAM
        plt.subplot(num_images, 2, 2 * i + 1)
        plt.imshow(img)
        plt.title(f"Original: {original_label}", fontsize=8)
        plt.axis('off')

        plt.subplot(num_images, 2, 2 * i + 2)
        plt.imshow(overlay[..., ::-1])  # BGR to RGB
        plt.title(f"Grad-CAM: {grad_cam_label}", fontsize=8)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# === Example Usage ===
# Use "Conv_1" for MobileNetV2, "block14_sepconv2_act" for Xception
gradcam_on_multiple_images(
    model=saved_model,
    test_dir= TEST_DATASET_PATH,  # Your test data root
    conv_layer_name="Conv_1",            # Last conv layer of MobileNetV2
    num_images=4                        # Feel free to adjust
)


In [ ]:
model.summary()